# PerfectPetClub - Data Engineering ETL Pipeline
This notebook  Extract, Transform and Loads supplied database.

### The pipeline:
Extracts records from a CSV file.

Cleans and standardises extracted data.

Exports the cleaned dataset in CSV and Parquet formats.

Displays the original and transformed datasets for validation.


## 1. Environment Setup and Dependencies

Import the required Python libraries and configure the project environment.

In [7]:
from pathlib import Path
import sys
import pandas as pd

sys.path.insert(0, str(Path("../src").resolve()))


%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### 3. CRM Data Extraction and Transformation

Execute the CRM ETL pipeline and display the raw, cleaned and feature-engineered datasets.

In [9]:

from crm import run_crm_etl

# Call the function to run  crm etl file 
crm_df, cleaned_crm_df,crm_features_df = run_crm_etl()

# Display the first five rows to confirm to check extraction and transformation 
display(crm_df.head())

display(cleaned_crm_df.head())

display(crm_features_df.head())

Missing or invalid dates: 0
CRM ETL and feature engineering completed successfully!


,customer_id,recieved_datatime,platform,type
0,55,2026-01-01 00:42:47.000,['e-mail'],complaint
1,49,2026-01-01 01:25:34.000,['live-chat'],query
2,44,2026-01-01 03:33:55.001,['telephone'],query
3,33,2026-01-01 05:42:16.002,['e-mail'],complaint
4,33,2026-01-01 06:25:03.002,['e-mail'],complaint


,customer_id,received_datetime,platform,type
0,55,2026-01-01 00:42:47.000,e-mail,complaint
1,49,2026-01-01 01:25:34.000,live-chat,query
2,44,2026-01-01 03:33:55.001,telephone,query
3,33,2026-01-01 05:42:16.002,e-mail,complaint
4,33,2026-01-01 06:25:03.002,e-mail,complaint


,customer_id,total_interactions,complaint_count
0,1,3,2
1,2,45,21
2,3,167,90
3,4,281,132
4,5,5,1


### 4. Customer Subscription ETL – Module Imports

Import the extraction, validation and transformation functions for customer subscription data.

In [10]:
from extract_customer_subscriptions import (
    extract_customer_subscriptions,
    validate_customer_data,
    transform_customer_features
)


print('Environment and modules loaded successfully!')

Environment and modules loaded successfully!


### 5. Extract Customer Subscription Data

Retrieve customer records from the SQLite database and inspect the extracted dataset.

In [11]:
DB_PATH = '../data/PerfectPetClub.db'

raw_customers = extract_customer_subscriptions(db_path=DB_PATH)

print(f'Extracted {len(raw_customers)} customer records.')
raw_customers.head(3)

2026-11-14 13:42:44,432 - INFO - Connecting to database: ../data/PerfectPetClub.db
2026-11-14 13:42:44,441 - INFO - Customer count verified: 221 records.
2026-11-14 13:42:44,442 - INFO - Successfully extracted 221 customer records.


Extracted 221 customer records.


,customer_id,customer_name,email,county,postcode,subscription_id,sub_start_date,sub_end_date,auto_renew,subscription_tier,delivery_service
0,1,Anahit Sargsyan,Sargsyan701@somemail.co.uk,Berkshire,SL6 1CD,1,2026-01-01 00:00:00.000,2026-01-31 00:00:00.000,0,SuperSaver,Economy72
1,2,Leyla Mammadova,Mammadova282@example.com,Oxfordshire,OX7 5GH,2,2026-01-01 00:00:00.000,2026-11-27 00:00:00.000,1,SuperSaver,Economy72
2,3,Sana Shah,Sana858@notrealemail.com,Worcestershire,DY10 2JK,3,2026-01-01 00:00:00.000,2026-11-27 00:00:00.000,1,Premium,SignedFor24


### 6. Customer Data Quality Validation

Perform automated validation checks to assess the completeness, consistency and integrity of customer records.

In [12]:
print("--- Validating Customer Data ---")

cust_dq = validate_customer_data(raw_customers)

print("Customer validation results:")
for check, result in cust_dq.items():
    print(f"{check}: {result}")

2026-11-14 13:42:45,966 - INFO - Executing automated Data Quality checks on customer subscriptions...
2026-11-14 13:42:45,976 - INFO - All Customer Data Quality checks PASSED successfully (5/5).


--- Validating Customer Data ---
Customer validation results:
null_checks: {'customer_id': 0, 'email': 0, 'subscription_id': 0, 'auto_renew': 0}
is_unique: True
valid_auto_renew: True
valid_dates: True
date_logic_valid: True


### 7. Negative Testing – Invalid Subscription Dates

Introduce an intentionally invalid date to test whether the validation function detects incorrect data.

In [13]:

 
# Test: Can our validation detect an invalid date?

test_data = raw_customers.copy()

# Deliberately insert an invalid subscription date
test_data.loc[0, 'sub_end_date'] = 'invalid-date'

try:
    validate_customer_data(test_data)
    print("TEST FAILED: Invalid date was accepted.")

except AssertionError as e:
    print("TEST PASSED: Invalid date was detected.")
    print(e)



 

2026-11-14 13:42:47,089 - INFO - Executing automated Data Quality checks on customer subscriptions...


TEST PASSED: Invalid date was detected.
DQ FAIL: Missing or invalid subscription dates.


### 8. Customer Data Transformation and Feature Engineering

Clean the customer records, derive analytical features and prepare the final customer-level dataset.

In [14]:
# Transform customer and subscription data
clean_customers = transform_customer_features(raw_customers)

# Prepare customer-only dataset
final_dataset = clean_customers.drop(
    columns=['customer_name', 'postcode', 'subscription_id']
)

print("Total customers:", len(final_dataset))
print("Dataset shape:", final_dataset.shape)

display(final_dataset.head())

2026-11-14 13:42:48,195 - INFO - Applying feature engineering on customer subscriptions...
2026-11-14 13:42:48,205 - INFO - Target Engineered: 86/221 customers churned (38.91%).


Total customers: 221
Dataset shape: (221, 11)


,customer_id,email,county,sub_start_date,sub_end_date,auto_renew,subscription_tier,delivery_service,is_churned,subscription_tenure_days,postcode_area
0,1,Sargsyan701@somemail.co.uk,Berkshire,2026-01-01,2026-01-31,0,SuperSaver,Economy72,1,30,SL
1,2,Mammadova282@example.com,Oxfordshire,2026-01-01,2026-11-27,1,SuperSaver,Economy72,0,330,OX
2,3,Sana858@notrealemail.com,Worcestershire,2026-01-01,2026-11-27,1,Premium,SignedFor24,0,330,DY
3,4,Omar.Farooq4@example.com,Kent,2026-01-01,2026-11-27,1,Premium,SignedFor24,0,330,TN
4,5,Alejandro357@somemail.co.uk,Hertfordshire,2026-01-01,2026-05-01,0,SuperSaver,Economy72,1,120,WD


### 9. Churn Analysis by Subscription Tier

Calculate customer churn rates for each subscription tier to identify patterns in customer retention.

In [15]:
# Analyse churn rate by subscription tier

tier_summary = final_dataset.groupby("subscription_tier")["is_churned"].agg(
    total_customers="count",
    churn_rate="mean"
).reset_index()

# Convert churn rate into percentage
tier_summary["churn_rate"] = (
    tier_summary["churn_rate"] * 100
).round(2)

# Display results
print("--- Churn Rate by Subscription Tier ---")
display(tier_summary)

--- Churn Rate by Subscription Tier ---


,subscription_tier,total_customers,churn_rate
0,Basic,77,33.77
1,Premium,47,40.43
2,SuperSaver,97,42.27


### 10. Load Transformed Customer Data into Parquet

Create the output directory and export the transformed customer dataset in Parquet format.

In [16]:
# Create output directory
output_dir = Path('../output')
output_dir.mkdir(parents=True, exist_ok=True)

parquet_path = output_dir / 'perfectpetclub_churn_features.parquet'

# Export to Parquet
final_dataset.to_parquet(parquet_path, index=False, engine='pyarrow')

print(f'✓ Successfully loaded {len(final_dataset)} records into Parquet format:')
print(f'  Location: {parquet_path.resolve()}')
print(f'  File size: {parquet_path.stat().st_size:,} bytes')


✓ Successfully loaded 221 records into Parquet format:
  Location: /Users/roshangauchan/Documents/DataEngineering/CW2/DataEngineeringPresentation/output/perfectpetclub_churn_features.parquet
  File size: 16,039 bytes


### 11. Verify Parquet Output

Reload the exported Parquet file and inspect its record count, columns and data types.

In [17]:
# Verification: Read back parquet file to guarantee integrity
df_verified = pd.read_parquet(parquet_path)
print(f'Parquet verification passed: {df_verified.shape[0]} rows, {df_verified.shape[1]} columns.')
print('\nDataset Columns:')
for col in df_verified.columns:
    print(f' - {col} ({df_verified[col].dtype})')

Parquet verification passed: 221 rows, 11 columns.

Dataset Columns:
 - customer_id (int64)
 - email (object)
 - county (object)
 - sub_start_date (datetime64[ns])
 - sub_end_date (datetime64[ns])
 - auto_renew (int64)
 - subscription_tier (object)
 - delivery_service (object)
 - is_churned (int64)
 - subscription_tenure_days (int64)
 - postcode_area (object)


### 12. Final Customer Data Integrity Tests

Verify customer uniqueness, record counts, churn calculations, subscription durations and exported dataset structure.

In [18]:
# Final verification of feature engineering and Parquet output

# Check 1: Customer records have not been duplicated
assert final_dataset['customer_id'].is_unique, "Duplicate customers found"

# Check 2: Customer count remains unchanged
assert len(final_dataset) == len(raw_customers), "Customer count mismatch"

# Check 3: Churn calculation is correct
expected_churn = (final_dataset['auto_renew'] == 0).astype(int)
assert (final_dataset['is_churned'] == expected_churn).all(), "Incorrect churn values"

# Check 4: Subscription durations are not negative
assert (final_dataset['subscription_tenure_days'] >= 0).all(), "Negative duration found"

# Check 5: Parquet file contains the expected records and columns
assert df_verified.shape == final_dataset.shape, "Parquet dataset shape mismatch"
assert list(df_verified.columns) == list(final_dataset.columns), "Parquet columns mismatch"

print("PASS: Customer records are unique")
print("PASS: Customer count is correct")
print("PASS: Churn calculations are correct")
print("PASS: Subscription durations are valid")
print("PASS: Parquet structure is verified")

PASS: Customer records are unique
PASS: Customer count is correct
PASS: Churn calculations are correct
PASS: Subscription durations are valid
PASS: Parquet structure is verified


### 13. Server Log ETL – Environment and Module Setup

Locate the project directory and import the extraction, transformation and loading functions from server.py.

In [19]:
from pathlib import Path
import importlib.util
import pandas as pd

# Find the project folder containing src/ and data/
current_dir = Path.cwd().resolve()
project_root = next(
    (folder for folder in [current_dir, *current_dir.parents]
     if (folder / 'src' / 'server.py').is_file()
     and (folder / 'data').is_dir()),
    None
)

if project_root is None:
    raise FileNotFoundError('Could not find the project folder with src/ and data/.')

# Import functions from the project's server.py
server_file = project_root / 'src' / 'server.py'
spec = importlib.util.spec_from_file_location('perfectpetclub_etl_server', server_file)
server_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(server_module)

extract_data = server_module.extract_data
transform_data = server_module.transform_data
load_data = server_module.load_data

print('ETL functions imported successfully.')

ETL functions imported successfully.


### 14. Server Log Input and Output Configuration

Define the source CSV file and destination directory for processed server log data.

In [20]:
#Configure Input and Output files
input_file = project_root / 'data' / 'PerfectPetClub_server_log.csv'
output_dir = project_root / 'output'
parquet_file = output_dir / 'perfectpetclub_customer_features.parquet'

if not input_file.is_file():
    raise FileNotFoundError(f'Input CSV not found: {input_file}')

output_dir.mkdir(parents=True, exist_ok=True)
print('Input file:', input_file)
print('Output folder:', output_dir)

Input file: /Users/roshangauchan/Documents/DataEngineering/CW2/DataEngineeringPresentation/data/PerfectPetClub_server_log.csv
Output folder: /Users/roshangauchan/Documents/DataEngineering/CW2/DataEngineeringPresentation/output


### 15. Server Log ETL Pipeline Execution

Execute extraction, transformation, Parquet loading and validation of customer-level features derived from server logs.

In [21]:
def run_pipeline(input_path, output_dir):
    # EXTRACT
    print('STEP 1 - EXTRACT')
    raw_data = extract_data(input_path)
    print('Raw data shape:', raw_data.shape)

    # TRANSFORM
    print('\nSTEP 2 - TRANSFORM')
    clean_data, feature_data, quality_report = transform_data(raw_data)
    print('Cleaned event data:', clean_data.shape)
    print('Customer feature data:', feature_data.shape)

    # LOAD - save the final dataset as a Parquet file
    print('\nSTEP 3 - LOAD')
    
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Save final customer-level dataset
    parquet_path = output_dir / 'perfectpetclub_final_dataset.parquet'
    
    parquet_path = load_data(feature_data, parquet_path)
    
    print('Saved final dataset as Parquet:', parquet_path)

    # VALIDATE saved customer-level Parquet
    print('\nSTEP 4 - VALIDATE')
    final_data = pd.read_parquet(parquet_path)
    assert final_data['customer_id'].is_unique
    assert len(final_data) == len(feature_data)
    assert 'churn_label' in final_data.columns
    print('Saved data validation passed.')
    print('Final customer records:', len(final_data))

    return final_data, clean_data, feature_data, quality_report

final_data, clean_data, feature_data, quality_report = run_pipeline(input_file, output_dir)

STEP 1 - EXTRACT
Extracted 67,249 rows from PerfectPetClub_server_log.csv
Raw data shape: (67249, 5)

STEP 2 - TRANSFORM
Transformed 67,249 clean events into 221 customer records
Cleaned event data: (67249, 12)
Customer feature data: (221, 21)

STEP 3 - LOAD
Saved 221 customer records to /Users/roshangauchan/Documents/DataEngineering/CW2/DataEngineeringPresentation/output/perfectpetclub_final_dataset.parquet
Saved final dataset as Parquet: /Users/roshangauchan/Documents/DataEngineering/CW2/DataEngineeringPresentation/output/perfectpetclub_final_dataset.parquet

STEP 4 - VALIDATE
Saved data validation passed.
Final customer records: 221


### 16. Server Log ETL Results and Quality Report

Display cleaned server events, customer-level features and the ETL quality report.

In [22]:
print('CLEANED EVENT DATA')
display(clean_data.head(10))

print('CUSTOMER-LEVEL FEATURE DATA')
display(feature_data.head(10))

print('ETL QUALITY REPORT')
quality_report_df = pd.DataFrame(
    list(quality_report.items()),
    columns=['quality_check', 'result']
)
display(quality_report_df)

CLEANED EVENT DATA


,datetime,session_id,ip,login,user_agent,event_time,customer_id,device_type,event_date,is_weekend,is_recent_7d,is_recent_30d
0,2026-01-01 02:08:21.000,session_cookie::63fec84dedaa4f2cbaa10b413b5a7d...,239.66.231.182,sipho928@somemail.co.uk,"""Mozilla/5.0 (Desktop/Laptop; Cipher CoreBook ...",2026-01-01 02:08:21.000,e93f5ba12a7daade7e1b68f4a3b3cf37d2e0c131d10a83...,other,2026-01-01,False,False,False
1,2026-01-01 02:51:08.001,session_cookie::64bcd32ed1494bd1af6b7a233cd638...,200.200.140.49,sienna.martin923@notrealemail.com,"""Mozilla/5.0 (Mobile; Pulse PulsePhone) AppleW...",2026-01-01 02:51:08.001,3d6ca41f0c98b3d616292085b88e849f9534a1e7d49162...,mobile,2026-01-01,False,False,False
2,2026-01-01 04:16:42.001,session_cookie::f848746c9c96422b9eb050eccacfeb...,38.219.218.103,v134@somemail.co.uk,"""Mozilla/5.0 (Mobile; Titan Pocket Go) AppleWe...",2026-01-01 04:16:42.001,c41e071e7055181c52623ff7cae3ba3f5797b4812f0259...,mobile,2026-01-01,False,False,False
3,2026-01-01 04:59:29.002,session_cookie::4589deeb679c4ae68f62a1f61af2db...,126.234.10.95,v134@somemail.co.uk,"""Mozilla/5.0 (Mobile; Titan Pocket Go) AppleWe...",2026-01-01 04:59:29.002,c41e071e7055181c52623ff7cae3ba3f5797b4812f0259...,mobile,2026-01-01,False,False,False
4,2026-01-01 07:07:50.003,session_cookie::2bc1377dd7a74949be4e0035137f5f...,17.99.243.42,io415@fakemail.co.uk,"""Mozilla/5.0 (Tablet; Lumina CorePad) AppleWeb...",2026-01-01 07:07:50.003,beb2723c6b81f32680e640a8c6c68e3ee3ca68250e3fed...,tablet,2026-01-01,False,False,False
5,2026-01-01 07:50:37.003,session_cookie::af34bdb485f94a49a24d374a97af7d...,97.68.10.163,efe_okoro493@somemail.co.uk,"""Mozilla/5.0 (Desktop/Laptop; Orion AirBook) A...",2026-01-01 07:50:37.003,9c497b385e0c084c5e18b86166523208fc9b3d100d60e6...,other,2026-01-01,False,False,False
6,2026-01-01 08:33:24.003,session_cookie::46e4f6f60eb14aea95be5027371182...,56.43.1.170,z_siddiqui928@example.com,"""Mozilla/5.0 (Desktop/Laptop; Nova Tech UltraB...",2026-01-01 08:33:24.003,fb67b3fefa22ad2715e81a8f6a6928ed4db98b39f86405...,other,2026-01-01,False,False,False
7,2026-01-01 09:16:11.003,session_cookie::64d12fbe7ff049548055682a2da229...,142.202.57.48,rosie_king32@somemail.co.uk,"""Mozilla/5.0 (Tablet; Element Slate Lite) Appl...",2026-01-01 09:16:11.003,5ef78c577b212e494536897f4637e3bfa837048ed3d30a...,tablet,2026-01-01,False,False,False
8,2026-01-01 09:58:58.004,session_cookie::b5a1450ea8d34c54acb4c1d20f53df...,18.142.160.251,qe742@notrealemail.com,"""Mozilla/5.0 (Tablet; Lumina AirTab) AppleWebK...",2026-01-01 09:58:58.004,31027993950bb0e69a367eb78d99485ab860ae02d8f8a4...,tablet,2026-01-01,False,False,False
9,2026-01-01 10:41:45.004,session_cookie::2bee12b7fea0426796883fdfdb40e7...,103.41.248.24,c.vargas505@fakemail.co.uk,"""Mozilla/5.0 (Mobile; Kinetix Spark Phone) App...",2026-01-01 10:41:45.004,266c25eb7c4a94f2dfacb348b61ef9cd0a48c0c6a9254c...,mobile,2026-01-01,False,False,False


CUSTOMER-LEVEL FEATURE DATA


,customer_id,first_activity,last_activity,days_since_last_activity,total_events,unique_sessions,unique_ips,active_days,events_last_7d,events_last_30d,...,mobile_event_ratio,desktop_event_ratio,tablet_event_ratio,dominant_device,churn_label,weekend_events,mobile_events,desktop_events,tablet_events,other_device_events
0,000498d9f3b0d7c5101eb0ec371bac2b50a6108e6e6900...,2026-07-28 18:51:41.998,2026-11-09 18:51:20.490,0,176,176,176,85,8,46,...,0.0,0.0,0.0,other,<NA>,49,0,0,0,176
1,006da6f3ce706ea27f12c668a7ae5ac00bb10071d82fc0...,2026-07-20 17:51:49.803,2026-10-09 17:25:50.743,32,65,65,65,41,0,0,...,0.0,0.0,0.0,other,<NA>,15,0,0,0,65
2,019654c28e94cbc5de1ae12768e941700a3b96090b70e8...,2026-05-23 18:00:46.413,2026-09-12 21:26:57.108,58,319,319,319,103,0,0,...,0.0,0.0,0.0,other,<NA>,102,0,0,0,319
3,04c2bb8a9e4295d789c90faaa4cedcee9e5605bc2edeeb...,2026-01-14 19:19:32.326,2026-11-09 20:15:20.494,0,747,747,747,267,13,48,...,0.0,0.0,0.0,other,<NA>,242,0,0,0,747
4,059231aacea2ff89c6f6e940e04975b4cb130d8798dd5c...,2026-07-02 22:42:54.384,2026-11-08 03:00:11.454,2,293,293,293,112,14,64,...,0.0,0.0,1.0,tablet,<NA>,88,0,0,293,0
5,059267e72b07c94549846f5eadb1903d6b4b3bfe008be9...,2026-09-21 19:26:08.317,2026-11-07 21:54:02.451,2,71,71,71,34,8,41,...,0.0,0.0,0.0,other,<NA>,14,0,0,0,71
6,05a1a176210d1601042c005c40718be956f72b40e0339b...,2026-07-22 19:13:07.855,2026-11-09 21:50:00.499,0,141,141,141,78,11,38,...,1.0,0.0,0.0,mobile,<NA>,41,141,0,0,0
7,06d07db03f83770b0cd97cef74fb93cf7a0b4f6f5711d7...,2026-01-01 19:28:25.015,2026-01-14 17:30:12.320,299,49,49,49,14,0,0,...,1.0,0.0,0.0,mobile,<NA>,15,49,0,0,0
8,082061a4e73e89522649b15e24519c912fc9989256d6b8...,2026-07-10 13:27:59.559,2026-11-10 17:23:59.509,0,336,336,336,117,18,71,...,1.0,0.0,0.0,mobile,<NA>,101,336,0,0,0
9,0a194a40942ba0355ca8bdad5a3842eb0315635bf032d8...,2026-06-12 20:11:06.899,2026-06-15 18:41:03.966,147,11,11,11,4,0,0,...,1.0,0.0,0.0,mobile,<NA>,9,11,0,0,0


ETL QUALITY REPORT


,quality_check,result
0,input_rows,67249
1,invalid_timestamp_rows,0
2,missing_or_blank_login_rows,0
3,rows_after_required_cleaning,67249
4,duplicate_rows_removed,0
5,clean_event_rows,67249
6,customer_rows,221
7,unique_customers,221
8,dataset_latest_timestamp,2026-11-10 17:29:19.510000
9,churn_label_available,False


In [ ]:
import sys
import importlib
import sqlite3
import pandas as pd

from pathlib import Path
from IPython.display import display

# Find the project root automatically
PROJECT_ROOT = Path.cwd().resolve()

if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "src").is_dir():
    raise FileNotFoundError("Cannot find the project's src folder")

DB_PATH = PROJECT_ROOT / "data" / "PerfectPetClub.db"
OUTPUT_DIR = PROJECT_ROOT / "output"

# Check database exists
if not DB_PATH.is_file():
    raise FileNotFoundError(f"Database not found: {DB_PATH}")

# Create output directory
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Make src importable
src_path = str(PROJECT_ROOT / "src")

if src_path not in sys.path:
    sys.path.insert(0, src_path)

print("Project root:", PROJECT_ROOT)
print("Database:", DB_PATH)
print("Output directory:", OUTPUT_DIR)

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("../data/PerfectPetClub.db")

tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table'",
    conn
)

display(tables)

In [ ]:
# Inspect sales, payments and reviews table structures

tables_to_check = [
    "sale",
    "sale_product",
    "payment",
    "product_review",
    "product"
]

for table in tables_to_check:
    print(f"\n========== {table.upper()} ==========")

    columns = pd.read_sql_query(
        f"PRAGMA table_info({table})",
        conn
    )

    display(columns[["name", "type"]])

In [ ]:
tables_to_check = [
    "sale",
    "sale_product",
    "payment",
    "product_review",
    "product"
]

for table in tables_to_check:
    print(f"\n===== {table.upper()} =====")

    columns = pd.read_sql_query(
        f"PRAGMA table_info({table})",
        conn
    )

    print(columns[["name", "type"]].to_string(index=False))

In [ ]:
# Check the sales table
print("===== SALE TABLE =====")

sale_columns = pd.read_sql_query(
    "PRAGMA table_info(sale)",
    conn
)

display(sale_columns[["name", "type"]])

# Check relationships between tables
for table in ["sale", "sale_product", "payment"]:
    print(f"\n===== {table.upper()} RELATIONSHIPS =====")

    foreign_keys = pd.read_sql_query(
        f"PRAGMA foreign_key_list({table})",
        conn
    )

    display(foreign_keys)

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from extract_sales_reviews import extract_sales_and_reviews

# Run Lakshmi's extraction
sales_reviews_df = extract_sales_and_reviews(
    db_path=PROJECT_ROOT / "data" / "PerfectPetClub.db"
)

print("Total customers:", len(sales_reviews_df))
print("Dataset shape:", sales_reviews_df.shape)

display(sales_reviews_df.head())

In [ ]:
import sys
import importlib
from pathlib import Path

PROJECT_ROOT = Path(
    r"C:\Users\DHANALAKSHI R\DataEngineeringPresentation"
)

sys.path.insert(0, str(PROJECT_ROOT / "src"))

import extract_sales_reviews
importlib.reload(extract_sales_reviews)

# Execute ETL pipeline
sales_reviews_df = extract_sales_reviews.run_pipeline(
    db_path=DB_PATH
)

print("Total customers:", len(sales_reviews_df))
print("Dataset shape:", sales_reviews_df.shape)

display(sales_reviews_df.head())

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path(
    db_path=DB_PATH
)

output_dir = PROJECT_ROOT / "output"

output_dir.mkdir(parents=True, exist_ok=True)

print("Output folder created successfully!")
print("Folder location:", output_dir)

In [ ]:
# Import teammate's ETL module
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import extract_sales_reviews
importlib.reload(extract_sales_reviews)

# Run extraction and validation
sales_reviews_df = extract_sales_reviews.run_pipeline(
    db_path=DB_PATH
)

print("Total customers:", len(sales_reviews_df))
print("Dataset shape:", sales_reviews_df.shape)
display(sales_reviews_df.head())

In [ ]:
output_dir = PROJECT_ROOT / "output"
output_dir.mkdir(parents=True, exist_ok=True)

parquet_path = output_dir / "sales_reviews_features.parquet"

sales_reviews_df.to_parquet(
    parquet_path,
    engine="pyarrow",
    index=False
)

print("Parquet saved successfully!")
print("File location:", parquet_path)

In [ ]:
import pandas as pd

verified_df = pd.read_parquet(parquet_path)

assert verified_df.equals(sales_reviews_df)
assert verified_df["customer_id"].is_unique

print("Parquet verification PASSED!")
print("Dataset shape:", verified_df.shape)